In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/playground-series-s6e6/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e6/train.csv
/kaggle/input/competitions/playground-series-s6e6/test.csv


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb

# 1. データの読み込み
# 上で確認したパスを正確に指定します
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e6/train.csv')
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e6/test.csv')

# 2. 前処理
# ターゲット変数（予測対象）の列名は 'class' です
target_col = 'class'

# GALAXY, STAR, QSO という文字列を 0, 1, 2 の数値に変換
le = LabelEncoder()
X = train.drop(columns=['id', target_col], errors='ignore')
y = le.fit_transform(train[target_col])
X_test = test.drop(columns=['id'], errors='ignore')

# カテゴリ変数を自動で検知してXGBoost用に型変換
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
for col in cat_cols:
    X[col] = X[col].astype('category')
    X_test[col] = X_test[col].astype('category')

num_classes = len(le.classes_)

# 3. 交差検証（5-Fold Cross Validation）の設計
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros((len(X), num_classes))
test_preds = np.zeros((len(X_test), num_classes))

# 4. XGBoost学習ループ
for fold, (train_idx, val_idx) in enumerate(folds.split(X, y)):
    print(f"\n--- Training Fold {fold+1} ---")
    X_train, y_train = X.iloc[train_idx], y[train_idx]
    X_val, y_val = X.iloc[val_idx], y[val_idx]
    
    # イーロンのサーバー並みに効率よく勾配を下るXGBoostのパラメーター
    model = xgb.XGBClassifier(
        n_estimators=1000,
        max_depth=6,
        learning_rate=0.05,
        objective='multi:softprob', # 多クラスの確率を出力
        num_class=num_classes,
        tree_method='hist',
        device='cuda',
        enable_categorical=True,
        early_stopping_rounds=30,
        random_state=42
    )
    
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=100
    )
    
    # 検証データとテストデータの予測確率を蓄積
    oof_preds[val_idx] = model.predict_proba(X_val)
    test_preds += model.predict_proba(X_test) / folds.n_splits

# 5. スコア（Accuracy）の確認
oof_pred_labels = np.argmax(oof_preds, axis=1)
cv_accuracy = accuracy_score(y, oof_pred_labels)
print(f"\n==========================================")
print(f"全体（OOF）の予測正解率 (Accuracy): {cv_accuracy:.5f}")
print(f"==========================================")

# 6. 提出用ファイル（submission.csv）の作成
sub = pd.DataFrame({'id': test['id']})
# 最も確率の高かった数値を、元の文字列（GALAXY等）に戻して代入
sub[target_col] = le.inverse_transform(np.argmax(test_preds, axis=1))
sub.to_csv("submission.csv", index=False)
print("提出用ファイル 'submission.csv' が正常に作成されました！")


--- Training Fold 1 ---
[0]	validation_0-mlogloss:0.82117
[100]	validation_0-mlogloss:0.12790
[200]	validation_0-mlogloss:0.10849
[300]	validation_0-mlogloss:0.10112
[400]	validation_0-mlogloss:0.09758
[500]	validation_0-mlogloss:0.09521
[600]	validation_0-mlogloss:0.09361
[700]	validation_0-mlogloss:0.09251
[800]	validation_0-mlogloss:0.09173
[900]	validation_0-mlogloss:0.09115
[999]	validation_0-mlogloss:0.09072


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [12:46:40] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)



--- Training Fold 2 ---
[0]	validation_0-mlogloss:0.82125
[100]	validation_0-mlogloss:0.12864
[200]	validation_0-mlogloss:0.10902
[300]	validation_0-mlogloss:0.10180
[400]	validation_0-mlogloss:0.09793
[500]	validation_0-mlogloss:0.09572
[600]	validation_0-mlogloss:0.09403
[700]	validation_0-mlogloss:0.09303
[800]	validation_0-mlogloss:0.09222
[900]	validation_0-mlogloss:0.09164
[999]	validation_0-mlogloss:0.09132

--- Training Fold 3 ---
[0]	validation_0-mlogloss:0.82123
[100]	validation_0-mlogloss:0.12964
[200]	validation_0-mlogloss:0.11039
[300]	validation_0-mlogloss:0.10317
[400]	validation_0-mlogloss:0.09914
[500]	validation_0-mlogloss:0.09691
[600]	validation_0-mlogloss:0.09525
[700]	validation_0-mlogloss:0.09432
[800]	validation_0-mlogloss:0.09366
[900]	validation_0-mlogloss:0.09319
[999]	validation_0-mlogloss:0.09287

--- Training Fold 4 ---
[0]	validation_0-mlogloss:0.82113
[100]	validation_0-mlogloss:0.12820
[200]	validation_0-mlogloss:0.10946
[300]	validation_0-mlogloss:0.1